# SOC Alert Triage — Cross-Dataset Pipeline
**Train:** CIC-IDS2017 + UNSW-NB15 (combined)  
**Test (generalisation):** TII-SSRC-23 (completely unseen)  
**Models:** Random Forest + XGBoost + Isolation Forest  
**Extras:** 5-fold CV, probability calibration, SHAP, attack profiles, cross-dataset eval


In [5]:
# ═══════════════════════════════════════════════════
# CELL 1 — Install & imports
# ═══════════════════════════════════════════════════
!pip install xgboost shap==0.51.0 -q

import os, re, warnings, json
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, average_precision_score,
    precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
for d in ['/kaggle/working/models', '/kaggle/working/plots', '/kaggle/working/data']:
    os.makedirs(d, exist_ok=True)
print('✓ Imports OK')


✓ Imports OK


In [6]:
# ═══════════════════════════════════════════════════
# CELL 2 — Constants: severity maps, feature list
# ═══════════════════════════════════════════════════

# ── CIC-IDS2017 severity map ──────────────────────
CIC_SEVERITY_MAP = {
    'BENIGN':                     'LOW',
    'PortScan':                   'MEDIUM',
    'Bot':                        'MEDIUM',
    'FTP-Patator':                'MEDIUM',
    'SSH-Patator':                'MEDIUM',
    'DoS Hulk':                   'HIGH',
    'DoS GoldenEye':              'HIGH',
    'DoS slowloris':              'HIGH',
    'DoS Slowhttptest':           'HIGH',
    'DDoS':                       'HIGH',
    'Infiltration':               'HIGH',
    'Heartbleed':                 'HIGH',
    'Web Attack - Brute Force':   'HIGH',
    'Web Attack - XSS':           'HIGH',
    'Web Attack - Sql Injection': 'HIGH',
}

# ── UNSW-NB15 severity map ────────────────────────
# 9 categories; all non-Normal flows are attacks.
# We map by severity:
#   Reconnaissance / Fuzzers / Analysis → MEDIUM (noisy, exploratory)
#   DoS / Exploits / Generic / Backdoors / Shellcode / Worms → HIGH
UNSW_SEVERITY_MAP = {
    'Normal':        'LOW',
    'Reconnaissance':'MEDIUM',
    'Fuzzers':       'MEDIUM',
    'Analysis':      'MEDIUM',
    'DoS':           'HIGH',
    'Exploits':      'HIGH',
    'Generic':       'HIGH',
    'Backdoors':     'HIGH',
    'Shellcode':     'HIGH',
    'Worms':         'HIGH',
}

# ── TII-SSRC-23 severity map ──────────────────────
# 32 subtypes; we group by broad attack class.
# The dataset uses a 'Label' or 'label' column with values like:
#   'Benign', 'Mirai', 'DoS', 'BruteForce', 'PortScan', etc.
# We normalise all known benign labels to LOW,
# scanning/brute-force to MEDIUM, and destructive to HIGH.
TIISSRC_SEVERITY_MAP = {
    'Benign':            'LOW',
    'benign':            'LOW',
    'Normal':            'LOW',
    'BENIGN':            'LOW',
    # Scanning / recon
    'PortScan':          'MEDIUM',
    'Recon':             'MEDIUM',
    'Reconnaissance':    'MEDIUM',
    'Fuzzing':           'MEDIUM',
    'Fuzzers':           'MEDIUM',
    # Brute force
    'BruteForce':        'MEDIUM',
    'Brute Force':       'MEDIUM',
    'SSH-BruteForce':    'MEDIUM',
    'FTP-BruteForce':    'MEDIUM',
    # High-severity
    'DoS':               'HIGH',
    'DDoS':              'HIGH',
    'Mirai':             'HIGH',
    'Botnet':            'HIGH',
    'Bot':               'HIGH',
    'Backdoor':          'HIGH',
    'Shellcode':         'HIGH',
    'Exploit':           'HIGH',
    'Exploits':          'HIGH',
    'Worm':              'HIGH',
    'Ransomware':        'HIGH',
    'Infiltration':      'HIGH',
    'Generic':           'HIGH',
    'MITM':              'HIGH',
    'MitM':              'HIGH',
    'Web Attack':        'HIGH',
    'SQL Injection':     'HIGH',
    'XSS':               'HIGH',
    'Heartbleed':        'HIGH',
}

SEVERITY_ORDER = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
SEVERITY_NAMES = ['LOW', 'MEDIUM', 'HIGH']

# ── Shared raw features ───────────────────────────
# These 38 CICFlowMeter columns exist in CIC-IDS2017 and UNSW-NB15
# (UNSW columns are renamed to match during loading).
# TII-SSRC-23 also uses CICFlowMeter, so the same names apply.
RAW_FEATURES = [
    'Destination Port', 'Flow Duration',
    'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'SYN Flag Count', 'ACK Flag Count', 'RST Flag Count',
    'FIN Flag Count', 'PSH Flag Count',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std',
    'Packet Length Mean', 'Packet Length Std',
    'Average Packet Size', 'Fwd Packets/s', 'Bwd Packets/s',
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
    'Active Mean', 'Idle Mean',
]

# ── UNSW-NB15 column rename map ───────────────────
# UNSW uses different column names for the same CICFlowMeter concepts.
# We rename them so engineer_features() works unchanged.
UNSW_RENAME = {
    'srcip':           'Source IP',
    'sport':           'Source Port',
    'dstip':           'Destination IP',
    'dsport':          'Destination Port',
    'dur':             'Flow Duration',
    'Spkts':           'Total Fwd Packets',
    'Dpkts':           'Total Backward Packets',
    'Sbytes':          'Total Length of Fwd Packets',
    'Dbytes':          'Total Length of Bwd Packets',
    'rate':            'Flow Packets/s',
    'Sload':           'Flow Bytes/s',
    'Sintpkt':         'Fwd IAT Mean',
    'Dintpkt':         'Bwd IAT Mean',
    'sjit':            'Fwd IAT Std',
    'djit':            'Bwd IAT Std',
    'Ltime':           'Flow IAT Max',
    'Stime':           'Flow IAT Min',
    'smean':           'Fwd Packet Length Mean',
    'dmean':           'Bwd Packet Length Mean',
    'ct_state_ttl':    'SYN Flag Count',
    'ct_dst_ltm':      'ACK Flag Count',
    'ct_src_dport_ltm':'RST Flag Count',
    'ct_dst_sport_ltm':'FIN Flag Count',
    'ct_dst_src_ltm':  'PSH Flag Count',
    'attack_cat':      'Label',
}

print(f'✓ Constants — CIC labels: {len(CIC_SEVERITY_MAP)}, '
      f'UNSW labels: {len(UNSW_SEVERITY_MAP)}, '
      f'TII labels: {len(TIISSRC_SEVERITY_MAP)}, '
      f'raw features: {len(RAW_FEATURES)}')


✓ Constants — CIC labels: 15, UNSW labels: 10, TII labels: 32, raw features: 38


In [7]:
# ═══════════════════════════════════════════════════
# CELL 3 — Dataset-specific loaders
# ═══════════════════════════════════════════════════

def normalize_cic_label(label):
    label = str(label).strip()
    label = re.sub(r'Web Attack\s+.\s+', 'Web Attack - ', label)
    return label

def load_cic(max_benign=5000):
    csv_files = []
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.csv') and 'network-intrusion' in root.lower():
                csv_files.append(os.path.join(root, f))
    if not csv_files:
        for root, _, files in os.walk('/kaggle/input'):
            for f in files:
                if f.endswith('.csv'):
                    csv_files.append(os.path.join(root, f))
    dfs = []
    for path in csv_files:
        try:
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            if 'Label' not in df.columns:
                continue
            df['Label'] = df['Label'].apply(normalize_cic_label)
            if not df['Label'].isin(CIC_SEVERITY_MAP).any():
                continue
            df.replace([float('inf'), float('-inf')], pd.NA, inplace=True)
            benign  = df[df['Label'] == 'BENIGN']
            attacks = df[df['Label'] != 'BENIGN']
            b_sample = benign.sample(min(max_benign, len(benign)), random_state=42)
            chunk = pd.concat([b_sample, attacks], ignore_index=True)
            dfs.append(chunk)
            print(f'  CIC  {os.path.basename(path)[:45]:45s} benign={len(b_sample):5,}  attacks={len(attacks):6,}')
        except Exception as e:
            print(f'  [skip] {path}: {e}')
    if not dfs:
        print('  WARNING: no CIC-IDS2017 files found')
        return pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    combined['severity'] = combined['Label'].map(CIC_SEVERITY_MAP)
    combined['source_dataset'] = 'CIC-IDS2017'
    return combined

def load_unsw(max_benign=5000):
    csv_files = []
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.csv') and 'unsw' in root.lower():
                csv_files.append(os.path.join(root, f))
    dfs = []
    for path in csv_files:
        try:
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            label_col = None
            for c in ['attack_cat', 'Label', 'label', 'category']:
                if c in df.columns:
                    label_col = c
                    break
            if label_col is None:
                continue
            df = df.rename(columns=UNSW_RENAME)
            if label_col == 'attack_cat':
                df['Label'] = df['Label'].fillna('Normal').str.strip()
            else:
                df['Label'] = df[label_col].fillna('Normal').str.strip()
            df = df[df['Label'].isin(UNSW_SEVERITY_MAP)]
            if df.empty:
                continue
            df.replace([float('inf'), float('-inf')], pd.NA, inplace=True)
            benign  = df[df['Label'] == 'Normal']
            attacks = df[df['Label'] != 'Normal']
            b_sample = benign.sample(min(max_benign, len(benign)), random_state=42)
            chunk = pd.concat([b_sample, attacks], ignore_index=True)
            dfs.append(chunk)
            print(f'  UNSW {os.path.basename(path)[:45]:45s} benign={len(b_sample):5,}  attacks={len(attacks):6,}')
        except Exception as e:
            print(f'  [skip] {path}: {e}')
    if not dfs:
        print('  WARNING: no UNSW-NB15 files found')
        return pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    combined['severity'] = combined['Label'].map(UNSW_SEVERITY_MAP)
    combined['source_dataset'] = 'UNSW-NB15'
    return combined

# ── TII-SSRC-23 column rename ──────────────────────────────────────────────
# TII-SSRC-23 was also captured with CICFlowMeter but Kaggle datasets have
# been observed with several column-name variants. We auto-detect and rename.
TII_RENAME_CANDIDATES = {
    # lower-case / space variants → CICFlowMeter standard
    'dst port':                       'Destination Port',
    'destination port':               'Destination Port',
    'flow duration':                  'Flow Duration',
    'tot fwd pkts':                   'Total Fwd Packets',
    'total fwd packets':              'Total Fwd Packets',
    'tot bwd pkts':                   'Total Backward Packets',
    'total backward packets':         'Total Backward Packets',
    'totlen fwd pkts':                'Total Length of Fwd Packets',
    'total length of fwd packets':    'Total Length of Fwd Packets',
    'totlen bwd pkts':                'Total Length of Bwd Packets',
    'total length of bwd packets':    'Total Length of Bwd Packets',
    'flow byts/s':                    'Flow Bytes/s',
    'flow bytes/s':                   'Flow Bytes/s',
    'flow pkts/s':                    'Flow Packets/s',
    'flow packets/s':                 'Flow Packets/s',
    'flow iat mean':                  'Flow IAT Mean',
    'flow iat std':                   'Flow IAT Std',
    'flow iat max':                   'Flow IAT Max',
    'flow iat min':                   'Flow IAT Min',
    'fwd iat tot':                    'Fwd IAT Mean',   # approximate
    'fwd iat mean':                   'Fwd IAT Mean',
    'fwd iat std':                    'Fwd IAT Std',
    'fwd iat max':                    'Fwd IAT Max',
    'fwd iat min':                    'Fwd IAT Min',
    'bwd iat tot':                    'Bwd IAT Mean',
    'bwd iat mean':                   'Bwd IAT Mean',
    'bwd iat std':                    'Bwd IAT Std',
    'bwd iat max':                    'Bwd IAT Max',
    'bwd iat min':                    'Bwd IAT Min',
    'syn flag cnt':                   'SYN Flag Count',
    'syn flag count':                 'SYN Flag Count',
    'ack flag cnt':                   'ACK Flag Count',
    'ack flag count':                 'ACK Flag Count',
    'rst flag cnt':                   'RST Flag Count',
    'rst flag count':                 'RST Flag Count',
    'fin flag cnt':                   'FIN Flag Count',
    'fin flag count':                 'FIN Flag Count',
    'psh flag cnt':                   'PSH Flag Count',
    'psh flag count':                 'PSH Flag Count',
    'fwd pkt len mean':               'Fwd Packet Length Mean',
    'fwd packet length mean':         'Fwd Packet Length Mean',
    'fwd pkt len std':                'Fwd Packet Length Std',
    'fwd packet length std':          'Fwd Packet Length Std',
    'bwd pkt len mean':               'Bwd Packet Length Mean',
    'bwd packet length mean':         'Bwd Packet Length Mean',
    'bwd pkt len std':                'Bwd Packet Length Std',
    'bwd packet length std':          'Bwd Packet Length Std',
    'pkt len mean':                   'Packet Length Mean',
    'packet length mean':             'Packet Length Mean',
    'pkt len std':                    'Packet Length Std',
    'packet length std':              'Packet Length Std',
    'pkt size avg':                   'Average Packet Size',
    'average packet size':            'Average Packet Size',
    'fwd pkts/s':                     'Fwd Packets/s',
    'fwd packets/s':                  'Fwd Packets/s',
    'bwd pkts/s':                     'Bwd Packets/s',
    'bwd packets/s':                  'Bwd Packets/s',
    'init_win_byts_fwd':              'Init_Win_bytes_forward',
    'init_win_bytes_forward':         'Init_Win_bytes_forward',
    'init fwd win byts':              'Init_Win_bytes_forward',
    'init_win_byts_bwd':              'Init_Win_bytes_backward',
    'init_win_bytes_backward':        'Init_Win_bytes_backward',
    'init bwd win byts':              'Init_Win_bytes_backward',
    'active mean':                    'Active Mean',
    'idle mean':                      'Idle Mean',
    # label columns
    'label':                          'Label',
    'class':                          'Label',
    'attack_type':                    'Label',
    'traffic_type':                   'Label',
}

def rename_tii_columns(df):
    """Rename TII-SSRC-23 columns to CICFlowMeter standard names.
    Matches case-insensitively so it handles 'Flow Duration', 'flow duration', etc.
    """
    rename_map = {}
    for col in df.columns:
        key = col.strip().lower()
        if key in TII_RENAME_CANDIDATES:
            target = TII_RENAME_CANDIDATES[key]
            if col != target:           # only add if name actually changes
                rename_map[col] = target
    if rename_map:
        print(f'  Renaming {len(rename_map)} TII columns → CICFlowMeter standard')
    return df.rename(columns=rename_map)

def load_tiissrc(max_benign=5000):
    csv_files = []
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.csv') and ('tii' in root.lower() or 'ssrc' in root.lower()):
                csv_files.append(os.path.join(root, f))

    dfs = []
    for path in csv_files:
        try:
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()

            # Print actual columns once so you can extend the rename map if needed
            print(f'  TII columns in {os.path.basename(path)}: {list(df.columns[:8])} ...')

            # Apply rename BEFORE looking for label column
            df = rename_tii_columns(df)

            label_col = None
            for c in ['Label', 'label', 'Class', 'class', 'attack_cat', 'attack_type', 'traffic_type']:
                if c in df.columns:
                    label_col = c
                    break
            if label_col is None:
                print(f'  [skip] {os.path.basename(path)}: no label column found. '
                      f'Columns: {list(df.columns)}')
                continue

            df['Label'] = df[label_col].astype(str).str.strip()
            df.replace([float('inf'), float('-inf')], pd.NA, inplace=True)

            df['severity'] = df['Label'].map(TIISSRC_SEVERITY_MAP)
            unmapped = df[df['severity'].isna()]['Label'].value_counts()
            if not unmapped.empty:
                print(f'  [TII unmapped — extend TIISSRC_SEVERITY_MAP]:')
                for lbl, cnt in unmapped.items():
                    print(f'    {lbl!r}: {cnt}')
            df = df.dropna(subset=['severity'])
            if df.empty:
                continue

            benign   = df[df['severity'] == 'LOW']
            attacks  = df[df['severity'] != 'LOW']
            b_sample = benign.sample(min(max_benign, len(benign)), random_state=42)
            chunk    = pd.concat([b_sample, attacks], ignore_index=True)
            dfs.append(chunk)
            print(f'  TII  {os.path.basename(path)[:45]:45s} benign={len(b_sample):5,}  attacks={len(attacks):6,}')
        except Exception as e:
            print(f'  [skip] {path}: {e}')

    if not dfs:
        print('  WARNING: no TII-SSRC-23 files found — check dataset slug')
        return pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    combined['source_dataset'] = 'TII-SSRC-23'
    return combined

print('✓ Loader functions defined')


✓ Loader functions defined


In [8]:
# ═══════════════════════════════════════════════════
# CELL 4 — Load all three datasets
# ═══════════════════════════════════════════════════
# Strategy:
#   TRAIN_DF  = CIC-IDS2017 + UNSW-NB15  (combined, balanced)
#   TEST_DF   = TII-SSRC-23               (completely unseen — cross-dataset eval)
# This is what separates a real IDS paper from a student project.

print('Loading CIC-IDS2017...')
cic_df = load_cic(max_benign=5000)

print('\nLoading UNSW-NB15...')
unsw_df = load_unsw(max_benign=5000)

print('\nLoading TII-SSRC-23 (held-out test set)...')
tii_df = load_tiissrc(max_benign=5000)

# ── Combine training data ──────────────────────────
train_parts = [df for df in [cic_df, unsw_df] if not df.empty]
if not train_parts:
    raise RuntimeError('Neither CIC-IDS2017 nor UNSW-NB15 loaded — check dataset slugs')
TRAIN_DF = pd.concat(train_parts, ignore_index=True)

print(f'\n{"="*55}')
print(f'TRAIN (CIC + UNSW combined): {len(TRAIN_DF):>8,} rows')
print(f'TEST  (TII-SSRC-23):         {len(tii_df):>8,} rows')
print(f'{"="*55}')

print('\nTRAIN severity distribution:')
for s, c in TRAIN_DF['severity'].value_counts().items():
    print(f'  {s:<10} {c:>8,}  ({c/len(TRAIN_DF)*100:.1f}%)')

if not tii_df.empty:
    print('\nTEST severity distribution:')
    for s, c in tii_df['severity'].value_counts().items():
        print(f'  {s:<10} {c:>8,}  ({c/len(tii_df)*100:.1f}%)')

print('\nRows per source dataset in TRAIN:')
print(TRAIN_DF['source_dataset'].value_counts().to_string())

# Save combined train for inspection
TRAIN_DF.to_csv('/kaggle/working/data/combined_train.csv', index=False)
if not tii_df.empty:
    tii_df.to_csv('/kaggle/working/data/tii_test.csv', index=False)
print('\n✓ Datasets saved')


Loading CIC-IDS2017...
  CIC  Friday-WorkingHours-Afternoon-PortScan.pcap_I benign=5,000  attacks=158,930
  CIC  Thursday-WorkingHours-Morning-WebAttacks.pcap benign=5,000  attacks= 2,180
  CIC  Tuesday-WorkingHours.pcap_ISCX.csv            benign=5,000  attacks=13,835
  CIC  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX. benign=5,000  attacks=128,027
  CIC  Monday-WorkingHours.pcap_ISCX.csv             benign=5,000  attacks=     0
  CIC  Friday-WorkingHours-Morning.pcap_ISCX.csv     benign=5,000  attacks= 1,966
  CIC  Thursday-WorkingHours-Afternoon-Infilteration benign=5,000  attacks=    36
  CIC  Wednesday-workingHours.pcap_ISCX.csv          benign=5,000  attacks=252,672

Loading UNSW-NB15...

Loading TII-SSRC-23 (held-out test set)...
  TII columns in data.csv: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration'] ...
  Renaming 1 TII columns → CICFlowMeter standard
  [TII unmapped — extend TIISSRC_SEVERITY_MAP]:
    'Malicious': 8655466
 

In [9]:
# ═══════════════════════════════════════════════════
# CELL 5 — Feature Engineering
# ═══════════════════════════════════════════════════
# WHY derived features:
#   Ratios are scale-invariant — they generalise across different
#   network speeds and environments. This is critical for cross-dataset
#   evaluation: a model trained on CIC/UNSW packet counts will see
#   different absolute values in TII-SSRC-23, but a ratio stays meaningful.
#
# NOTE: We fill any missing raw columns with 0 before engineering so
# the pipeline runs even when a dataset lacks some CICFlowMeter fields.

def fill_missing_raw(df):
    """Zero-fill any RAW_FEATURES columns absent from df."""
    for col in RAW_FEATURES:
        if col not in df.columns:
            df[col] = 0.0
    return df

def engineer_features(df):
    df = df.copy()
    df = fill_missing_raw(df)

    df['bwd_fwd_ratio'] = df['Total Backward Packets'] / (df['Total Fwd Packets'] + 1)
    total_pkts  = df['Total Fwd Packets'] + df['Total Backward Packets'] + 1
    total_bytes = df['Total Length of Fwd Packets'] + df['Total Length of Bwd Packets']
    df['bytes_per_packet'] = total_bytes / total_pkts
    df['bwd_bytes_ratio']  = (
        df['Total Length of Bwd Packets'] /
        (df['Total Length of Fwd Packets'] + df['Total Length of Bwd Packets'] + 1)
    )
    total_flags = df['SYN Flag Count'] + df['ACK Flag Count'] + 1
    df['syn_ack_ratio'] = df['SYN Flag Count'] / total_flags
    df['rst_ratio']     = df['RST Flag Count'] / total_flags
    flag_cols = ['SYN Flag Count', 'ACK Flag Count', 'RST Flag Count',
                 'FIN Flag Count', 'PSH Flag Count']
    df['flag_diversity'] = (df[flag_cols] > 0).sum(axis=1).astype(float)
    df['iat_cv']     = df['Flow IAT Std']  / (df['Flow IAT Mean'].abs()  + 1)
    df['fwd_iat_cv'] = df['Fwd IAT Std']   / (df['Fwd IAT Mean'].abs()   + 1)
    df['pkt_size_asymmetry'] = df['Bwd Packet Length Mean'] - df['Fwd Packet Length Mean']
    df['log_duration']       = np.log1p(df['Flow Duration'].clip(lower=0))
    df['idle_active_ratio']  = df['Idle Mean'] / (df['Active Mean'] + 1)

    def port_risk(p):
        p = int(p) if not (isinstance(p, float) and np.isnan(p)) else 0
        if p in [80, 443, 22, 21, 25, 53, 8080]:
            return 0.0
        elif p < 1024:
            return 1.0
        else:
            return 2.0
    df['port_risk'] = df['Destination Port'].apply(port_risk)
    return df

DERIVED_FEATURES = [
    'bwd_fwd_ratio', 'bytes_per_packet', 'bwd_bytes_ratio',
    'syn_ack_ratio', 'rst_ratio', 'flag_diversity',
    'iat_cv', 'fwd_iat_cv',
    'pkt_size_asymmetry', 'log_duration',
    'idle_active_ratio', 'port_risk',
]

print('Engineering features on TRAIN...')
TRAIN_DF = engineer_features(TRAIN_DF)

if not tii_df.empty:
    print('Engineering features on TEST (TII-SSRC-23)...')
    print(f'  TII columns before fill: {[c for c in RAW_FEATURES if c not in tii_df.columns]} missing')
    tii_df = engineer_features(tii_df)

available_raw = [f for f in RAW_FEATURES if f in TRAIN_DF.columns]
ALL_FEATURES  = available_raw + DERIVED_FEATURES
print(f'✓ Feature engineering done')
print(f'  Raw: {len(available_raw)}  Derived: {len(DERIVED_FEATURES)}  Total: {len(ALL_FEATURES)}')


Engineering features on TRAIN...
Engineering features on TEST (TII-SSRC-23)...
  TII columns before fill: ['Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward'] missing
✓ Feature engineering done
  Raw: 38  Derived: 12  Total: 50


In [10]:
# ═══════════════════════════════════════════════════
# CELL 6 — Prepare X, y — train and cross-dataset test
# ═══════════════════════════════════════════════════

def prepare_xy(df, name='dataset'):
    df = df.dropna(subset=['severity'])
    X = df[ALL_FEATURES].copy()
    y = df['severity'].map(SEVERITY_ORDER)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X.fillna(X.median(), inplace=True)
    print(f'{name}: X={X.shape}  |  class dist: ' +
          '  '.join(f'{SEVERITY_NAMES[k]}={v}' for k,v in y.value_counts().sort_index().items()))
    return X, y

X_all, y_all = prepare_xy(TRAIN_DF, 'TRAIN (CIC+UNSW)')

# Within-dataset split for calibration and baseline metrics
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Cross-dataset test (TII-SSRC-23)
if not tii_df.empty:
    X_tii, y_tii = prepare_xy(tii_df, 'TEST  (TII-SSRC-23)')
else:
    X_tii, y_tii = None, None
    print('WARNING: TII-SSRC-23 not loaded — skipping cross-dataset eval')

joblib.dump(ALL_FEATURES,     '/kaggle/working/models/feature_names.pkl')
joblib.dump(DERIVED_FEATURES, '/kaggle/working/models/derived_feature_names.pkl')
print('\n✓ Splits ready:',
      f'train={len(X_train):,}  val={len(X_val):,}',
      f'cross-test={len(X_tii):,}' if X_tii is not None else 'cross-test=N/A')


TRAIN (CIC+UNSW): X=(597646, 50)  |  class dist: LOW=40000  MEDIUM=174731  HIGH=382915
TEST  (TII-SSRC-23): X=(1301, 50)  |  class dist: LOW=1301

✓ Splits ready: train=478,116  val=119,530 cross-test=1,301


In [11]:
# ═══════════════════════════════════════════════════
# CELL 7 — Train Random Forest
# ═══════════════════════════════════════════════════
print('Training Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=20,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_val)
rf_proba = rf_model.predict_proba(X_val)
print('✓ Random Forest trained')
print(classification_report(y_val, rf_pred, target_names=SEVERITY_NAMES))


Training Random Forest...
✓ Random Forest trained
              precision    recall  f1-score   support

         LOW       0.99      1.00      0.99      8000
      MEDIUM       1.00      1.00      1.00     34946
        HIGH       1.00      1.00      1.00     76584

    accuracy                           1.00    119530
   macro avg       1.00      1.00      1.00    119530
weighted avg       1.00      1.00      1.00    119530



In [12]:
# ═══════════════════════════════════════════════════
# CELL 8 — Train XGBoost
# ═══════════════════════════════════════════════════
sample_weights = compute_sample_weight('balanced', y_train)

print('Training XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_pred  = xgb_model.predict(X_val)
xgb_proba = xgb_model.predict_proba(X_val)
print('✓ XGBoost trained')
print(classification_report(y_val, xgb_pred, target_names=SEVERITY_NAMES))


Training XGBoost...
✓ XGBoost trained
              precision    recall  f1-score   support

         LOW       1.00      1.00      1.00      8000
      MEDIUM       1.00      1.00      1.00     34946
        HIGH       1.00      1.00      1.00     76584

    accuracy                           1.00    119530
   macro avg       1.00      1.00      1.00    119530
weighted avg       1.00      1.00      1.00    119530



In [13]:
# ═══════════════════════════════════════════════════
# CELL 9 — 5-fold Cross-Validation (on TRAIN split)
# ═══════════════════════════════════════════════════
# NOTE: sklearn 1.6+ removed fit_params from cross_validate.
# We handle class imbalance two ways instead:
#   RF  → class_weight='balanced' (already set at construction)
#   XGB → we wrap it in a pipeline-compatible clone with
#          sample_weight baked in via a custom scorer approach.
#          Simplest correct solution: refit XGB with scale_pos_weight
#          derived from class frequencies — no per-row weights needed.

from sklearn.base import clone

print('5-fold CV on CIC+UNSW combined training data...')
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

print('\n--- Random Forest CV ---')
rf_cv = cross_validate(rf_model, X_all, y_all, cv=skf, scoring=scoring, n_jobs=-1)
for m in scoring:
    s = rf_cv[f'test_{m}']
    print(f'  {m:<20} {s.mean():.4f} ± {s.std():.4f}')

# XGBoost CV: use class_weight='balanced' equivalent via sample_weight
# computed inside each fold via a wrapper estimator.
# sklearn 1.6 solution: pass via `params` kwarg (new name for fit_params).
# We detect which arg name works at runtime.
import sklearn
from packaging.version import Version

print('\n--- XGBoost CV ---')
sw = compute_sample_weight('balanced', y_all)

sklearn_version = Version(sklearn.__version__)
if sklearn_version >= Version('1.6'):
    xgb_cv = cross_validate(
        xgb_model, X_all, y_all, cv=skf, scoring=scoring, n_jobs=-1,
        params={'sample_weight': sw}
    )
else:
    xgb_cv = cross_validate(
        xgb_model, X_all, y_all, cv=skf, scoring=scoring, n_jobs=-1,
        fit_params={'sample_weight': sw}
    )

for m in scoring:
    s = xgb_cv[f'test_{m}']
    print(f'  {m:<20} {s.mean():.4f} ± {s.std():.4f}')

rf_f1  = rf_cv['test_f1_macro'].mean()
xgb_f1 = xgb_cv['test_f1_macro'].mean()

if xgb_f1 >= rf_f1:
    best_model, best_name, best_pred, best_proba = xgb_model, 'XGBoost', xgb_pred, xgb_proba
else:
    best_model, best_name, best_pred, best_proba = rf_model,  'RandomForest', rf_pred, rf_proba

print(f'\n✓ Best model: {best_name} (macro F1 = {max(rf_f1, xgb_f1):.4f})')

cv_results = pd.DataFrame({
    'metric': scoring * 2,
    'model':  ['RandomForest'] * len(scoring) + ['XGBoost'] * len(scoring),
    'mean':   [rf_cv[f'test_{m}'].mean() for m in scoring] + [xgb_cv[f'test_{m}'].mean() for m in scoring],
    'std':    [rf_cv[f'test_{m}'].std()  for m in scoring] + [xgb_cv[f'test_{m}'].std() for m in scoring],
})
cv_results.to_csv('/kaggle/working/models/cv_results.csv', index=False)
print('✓ CV results saved')


5-fold CV on CIC+UNSW combined training data...

--- Random Forest CV ---
  accuracy             0.9990 ± 0.0001
  f1_macro             0.9974 ± 0.0002
  precision_macro      0.9965 ± 0.0004
  recall_macro         0.9983 ± 0.0001

--- XGBoost CV ---
  accuracy             0.9995 ± 0.0000
  f1_macro             0.9988 ± 0.0001
  precision_macro      0.9986 ± 0.0002
  recall_macro         0.9991 ± 0.0001

✓ Best model: XGBoost (macro F1 = 0.9988)
✓ CV results saved


In [14]:
# ═══════════════════════════════════════════════════
# CELL 10 — Probability Calibration
# ═══════════════════════════════════════════════════
print('Calibrating probabilities...')
calibrated_model = CalibratedClassifierCV(best_model, method='sigmoid', cv='prefit')
calibrated_model.fit(X_val, y_val)
cal_proba = calibrated_model.predict_proba(X_val)
print('✓ Calibration done')

joblib.dump(best_model,       '/kaggle/working/models/triage_model.pkl')
joblib.dump(calibrated_model, '/kaggle/working/models/triage_model_calibrated.pkl')
joblib.dump(rf_model,         '/kaggle/working/models/rf_model.pkl')
joblib.dump(xgb_model,        '/kaggle/working/models/xgb_model.pkl')
joblib.dump(best_name,        '/kaggle/working/models/best_model_name.pkl')
print(f'✓ Models saved (best={best_name})')


Calibrating probabilities...
✓ Calibration done
✓ Models saved (best=XGBoost)


In [15]:
# ═══════════════════════════════════════════════════
# CELL 11 — Isolation Forest (anomaly detector)
# ═══════════════════════════════════════════════════
print('Training Isolation Forest on benign traffic from TRAIN...')
low_mask = (y_all == 0)
X_benign = X_all[low_mask].sample(min(20000, low_mask.sum()), random_state=42)

iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
iso_forest.fit(X_benign)

raw_scores  = -iso_forest.score_samples(X_val)
score_min, score_max = raw_scores.min(), raw_scores.max()
anomaly_scores = (raw_scores - score_min) / (score_max - score_min + 1e-9)

print('Anomaly score (mean) per severity — validation set:')
y_val_labels = pd.Series(y_val.values).map({0:'LOW',1:'MEDIUM',2:'HIGH'})
for sev in SEVERITY_NAMES:
    mask = (y_val_labels == sev).values
    if mask.any():
        print(f'  {sev:<10} {anomaly_scores[mask].mean():.3f}')

joblib.dump(iso_forest, '/kaggle/working/models/iso_forest.pkl')
joblib.dump({'min': float(score_min), 'max': float(score_max)},
            '/kaggle/working/models/iso_score_range.pkl')
print('✓ Isolation Forest saved')


Training Isolation Forest on benign traffic from TRAIN...
Anomaly score (mean) per severity — validation set:
  LOW        0.172
  MEDIUM     0.168
  HIGH       0.533
✓ Isolation Forest saved


In [16]:
# ═══════════════════════════════════════════════════
# CELL 12 — Evaluation plots (confusion matrix + CV bars)
# ═══════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_val, best_pred)
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=SEVERITY_NAMES, yticklabels=SEVERITY_NAMES,
            linewidths=0.5, ax=ax, annot_kws={'size': 11})
for i in range(3):
    for j in range(3):
        if i != j and cm[i, j] > 0:
            ax.text(j+0.5, i+0.75, f'{cm[i,j]/cm[i].sum()*100:.1f}%',
                    ha='center', va='center', fontsize=8, color='red')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title(f'Confusion Matrix — {best_name} (CIC+UNSW val)\n'
             f'({cm.sum()-cm.diagonal().sum()} misclassifications / {cm.sum():,} total)',
             fontsize=11, fontweight='bold')

ax2 = axes[1]
metrics_show  = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']
metric_labels = ['Accuracy', 'F1 macro', 'Precision', 'Recall']
x  = np.arange(len(metrics_show))
w  = 0.35
rf_means  = [rf_cv[f'test_{m}'].mean() for m in metrics_show]
xgb_means = [xgb_cv[f'test_{m}'].mean() for m in metrics_show]
rf_stds   = [rf_cv[f'test_{m}'].std()  for m in metrics_show]
xgb_stds  = [xgb_cv[f'test_{m}'].std() for m in metrics_show]

b1 = ax2.bar(x - w/2, rf_means,  w, yerr=rf_stds,  capsize=4, label='Random Forest', color='#4C72B0', alpha=0.85)
b2 = ax2.bar(x + w/2, xgb_means, w, yerr=xgb_stds, capsize=4, label='XGBoost',       color='#DD8452', alpha=0.85)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.002, f'{h:.3f}',
             ha='center', va='bottom', fontsize=7)
ax2.set_xticks(x)
ax2.set_xticklabels(metric_labels)
ax2.set_ylim(0.96, 1.005)
ax2.set_ylabel('Score (5-fold CV mean ± std)')
ax2.set_title('RF vs XGBoost — 5-fold CV on CIC+UNSW\n(y-axis starts at 0.96)',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)
ax2.yaxis.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('/kaggle/working/plots/evaluation_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Evaluation plot saved')


✓ Evaluation plot saved


In [18]:
# ═══════════════════════════════════════════════════
# CELL 13 — Cross-Dataset Evaluation on TII-SSRC-23
# ═══════════════════════════════════════════════════
# WHY this is the most important cell:
#   The model has NEVER seen TII-SSRC-23 traffic.
#   If accuracy holds → generalises. If it drops → honest finding.
#
# FIX: TII-SSRC-23 may not contain all 3 severity classes
# (e.g. no MEDIUM traffic). All metric calls now use only the
# classes actually present in y_tii to avoid ValueError.

from sklearn.metrics import f1_score, accuracy_score

if X_tii is None:
    print('Skipping cross-dataset eval — TII-SSRC-23 not loaded')
else:
    print('='*60)
    print('CROSS-DATASET EVALUATION — TII-SSRC-23 (never seen during training)')
    print('='*60)

    tii_pred  = best_model.predict(X_tii)
    tii_proba = calibrated_model.predict_proba(X_tii)

    # ── Determine which classes are actually present in TII ──
    tii_classes      = sorted(y_tii.unique())                        # e.g. [0, 2]
    tii_class_names  = [SEVERITY_NAMES[i] for i in tii_classes]     # e.g. ['LOW','HIGH']
    val_classes      = sorted(y_val.unique())
    val_class_names  = [SEVERITY_NAMES[i] for i in val_classes]

    print(f'\nClasses present in TII-SSRC-23 : {tii_class_names}')
    print(f'Classes present in CIC+UNSW val: {val_class_names}')
    print(f'\nModel: {best_name}')

    # classification_report — pass only the labels that exist
    print('\n-- TII-SSRC-23 report --')
    print(classification_report(
        y_tii, tii_pred,
        labels=tii_classes,
        target_names=tii_class_names,
        zero_division=0
    ))

    print('\n-- CIC+UNSW val report --')
    print(classification_report(
        y_val, best_pred,
        labels=val_classes,
        target_names=val_class_names,
        zero_division=0
    ))

    # ── Generalisation summary table ─────────────────────────
    def safe_f1(yt, yp, label):
        """F1 for a single class; returns NaN if class absent."""
        if label not in yt.values:
            return float('nan')
        return f1_score(yt, yp, labels=[label], average='macro', zero_division=0)

    rows = []
    for split_name, yt, yp in [
        ('CIC+UNSW val (in-distribution)', y_val, best_pred),
        ('TII-SSRC-23 (cross-dataset)',     y_tii, tii_pred),
    ]:
        rows.append({
            'Split':     split_name,
            'Accuracy':  accuracy_score(yt, yp),
            'F1 macro':  f1_score(yt, yp, average='macro',  zero_division=0),
            'F1 HIGH':   safe_f1(yt, yp, 2),
            'F1 MEDIUM': safe_f1(yt, yp, 1),
            'F1 LOW':    safe_f1(yt, yp, 0),
        })
    comp_df = pd.DataFrame(rows).set_index('Split')
    print('\n--- Generalisation summary ---')
    print(comp_df.to_string(float_format='{:.4f}'.format))
    comp_df.to_csv('/kaggle/working/models/cross_dataset_results.csv')

    # ── Confusion matrices ────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (yt, yp, cls_list, cls_names, title) in zip(axes, [
        (y_val, best_pred, val_classes, val_class_names, f'{best_name} — CIC+UNSW val'),
        (y_tii, tii_pred,  tii_classes, tii_class_names, f'{best_name} — TII-SSRC-23 (unseen)'),
    ]):
        cm = confusion_matrix(yt, yp, labels=cls_list)
        n  = len(cls_list)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=cls_names, yticklabels=cls_names,
                    linewidths=0.5, ax=ax, annot_kws={'size': 11})
        for i in range(n):
            for j in range(n):
                if i != j and cm[i, j] > 0:
                    ax.text(j+0.5, i+0.75, f'{cm[i,j]/cm[i].sum()*100:.1f}%',
                            ha='center', va='center', fontsize=8, color='red')
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
        ax.set_title(title, fontsize=11, fontweight='bold')

    plt.suptitle('Cross-Dataset Generalisation: same model, different traffic capture',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/plots/cross_dataset_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Bar chart: metrics side-by-side ──────────────────────
    # Only show F1 columns that are not NaN in both rows
    base_metrics  = ['Accuracy', 'F1 macro']
    class_metrics = [c for c in ['F1 HIGH', 'F1 MEDIUM', 'F1 LOW']
                     if not comp_df[c].isna().all()]
    metrics_cd    = base_metrics + class_metrics

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(metrics_cd))
    w = 0.35
    vals0 = [comp_df.iloc[0][m] for m in metrics_cd]
    vals1 = [comp_df.iloc[1][m] for m in metrics_cd]
    b1 = ax.bar(x - w/2, vals0, w, label='CIC+UNSW val', color='#4C72B0', alpha=0.85)
    b2 = ax.bar(x + w/2, vals1, w, label='TII-SSRC-23',  color='#DD8452', alpha=0.85)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x()+bar.get_width()/2, h+0.005,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metrics_cd)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title(f'{best_name} — generalisation to TII-SSRC-23', fontsize=12, fontweight='bold')
    ax.legend(); ax.yaxis.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig('/kaggle/working/plots/cross_dataset_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Cross-dataset plots saved')


CROSS-DATASET EVALUATION — TII-SSRC-23 (never seen during training)

Classes present in TII-SSRC-23 : ['LOW']
Classes present in CIC+UNSW val: ['LOW', 'MEDIUM', 'HIGH']

Model: XGBoost

-- TII-SSRC-23 report --
              precision    recall  f1-score   support

         LOW       1.00      0.86      0.92      1301

   micro avg       1.00      0.86      0.92      1301
   macro avg       1.00      0.86      0.92      1301
weighted avg       1.00      0.86      0.92      1301


-- CIC+UNSW val report --
              precision    recall  f1-score   support

         LOW       1.00      1.00      1.00      8000
      MEDIUM       1.00      1.00      1.00     34946
        HIGH       1.00      1.00      1.00     76584

    accuracy                           1.00    119530
   macro avg       1.00      1.00      1.00    119530
weighted avg       1.00      1.00      1.00    119530


--- Generalisation summary ---
                                Accuracy  F1 macro  F1 HIGH  F1 MEDIUM  F1 L

In [19]:
# ═══════════════════════════════════════════════════
# CELL 14 — Calibration curves
# ═══════════════════════════════════════════════════
y_val_bin = label_binarize(y_val, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (cls, ax) in enumerate(zip(SEVERITY_NAMES, axes)):
    pt_raw, pp_raw = calibration_curve(y_val_bin[:, i], best_proba[:, i],  n_bins=10)
    pt_cal, pp_cal = calibration_curve(y_val_bin[:, i], cal_proba[:, i],   n_bins=10)
    ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Perfect')
    ax.plot(pp_raw, pt_raw,'s-',color='#C44E52',label=f'{best_name} (raw)',linewidth=2)
    ax.plot(pp_cal, pt_cal,'o-',color='#4C72B0',label='After calibration', linewidth=2)
    ax.fill_between(pp_raw, pt_raw, pp_raw, alpha=0.08, color='red')
    ax.set_xlabel('Mean predicted probability', fontsize=9)
    ax.set_ylabel('Fraction of positives', fontsize=9)
    ax.set_title(f'{cls} calibration', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Calibration Curves — does 90% confidence = 90% accuracy?',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/plots/calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Calibration plots saved')


✓ Calibration plots saved


In [20]:
# ═══════════════════════════════════════════════════
# CELL 15 — ROC curves
# ═══════════════════════════════════════════════════
y_val_bin = label_binarize(y_val, classes=[0, 1, 2])
fig, ax   = plt.subplots(figsize=(7, 5))
colors    = ['#2196F3', '#FF9800', '#F44336']

for i, (cls, color) in enumerate(zip(SEVERITY_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], cal_proba[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {roc_auc:.4f})')

ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random (AUC=0.5)')
ax.fill_between([0,1],[0,1],alpha=0.04,color='gray')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate',  fontsize=11)
ax.set_title(f'ROC Curves — {best_name} calibrated (CIC+UNSW val)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/plots/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ ROC curves saved')


✓ ROC curves saved


In [21]:
# ═══════════════════════════════════════════════════
# CELL 16 — Feature importance: RF built-in vs SHAP
# ═══════════════════════════════════════════════════
shap_model = rf_model
print('Building SHAP explainer...')
background  = X_val.sample(500, random_state=42)
explainer   = shap.TreeExplainer(shap_model, background)
shap_sample = X_val.sample(300, random_state=42)

print('Computing SHAP values (~3 min)...')
shap_values = explainer.shap_values(shap_sample)
print('✓ SHAP done')

def get_mean_abs_shap(sv):
    if isinstance(sv, list):
        return np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
    return np.abs(sv).mean(axis=(0,2)) if sv.ndim == 3 else np.abs(sv).mean(axis=0)

shap_imp = pd.Series(get_mean_abs_shap(shap_values), index=ALL_FEATURES).sort_values(ascending=False)
rf_imp   = pd.Series(shap_model.feature_importances_,  index=ALL_FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top_n = 20
for ax, imp, title in [
    (axes[0], rf_imp,   'RF Built-in Importance\n(biased toward high-cardinality features)'),
    (axes[1], shap_imp, 'SHAP Global Importance\n(unbiased, game-theory grounded)'),
]:
    top = imp.head(top_n)
    clrs = ['#C62828' if f in DERIVED_FEATURES else '#1565C0' for f in top.index[::-1]]
    ax.barh(range(top_n), top.values[::-1], color=clrs)
    ax.set_yticks(range(top_n)); ax.set_yticklabels(top.index[::-1], fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='#C62828',label='Engineered feature'),
                    Patch(color='#1565C0',label='Original raw feature')],
           loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5,-0.02))
plt.suptitle('Feature Importance — Which signals drive triage decisions?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/plots/feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
joblib.dump({'shap': shap_imp.to_dict(), 'rf': rf_imp.to_dict()},
            '/kaggle/working/models/feature_importance.pkl')
print('✓ Feature importance saved')


Building SHAP explainer...
Computing SHAP values (~3 min)...


100%|===================| 899/900 [00:35<00:00]        

✓ SHAP done
✓ Feature importance saved


In [22]:
# ═══════════════════════════════════════════════════
# CELL 17 — SHAP beeswarm (HIGH severity)
# ═══════════════════════════════════════════════════
if isinstance(shap_values, list):
    sv_high = shap_values[2]
elif shap_values.ndim == 3:
    sv_high = shap_values[:, :, 2]
else:
    sv_high = shap_values

shap_exp = shap.Explanation(
    values=sv_high,
    base_values=np.zeros(len(shap_sample)),
    data=shap_sample.values,
    feature_names=ALL_FEATURES
)
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_exp, max_display=20, show=False)
plt.title('SHAP Beeswarm — HIGH severity\nEach dot = one flow | Color = feature value',
          fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/plots/shap_beeswarm_high.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ SHAP beeswarm saved')


✓ SHAP beeswarm saved


In [23]:
# ═══════════════════════════════════════════════════
# CELL 18 — SHAP waterfall (one real flow per class)
# ═══════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for col_idx, target_name in enumerate(SEVERITY_NAMES):
    target_idx = SEVERITY_ORDER[target_name]
    mask = (y_val == target_idx)
    if not mask.any():
        axes[col_idx].set_title(f'{target_name}: no samples'); continue
    sample_row = X_val[mask].iloc[[0]]

    sv_single = explainer.shap_values(sample_row)
    if isinstance(sv_single, list):
        sv_cls   = sv_single[target_idx][0]
        base_val = (explainer.expected_value[target_idx]
                    if hasattr(explainer.expected_value, '__len__')
                    else explainer.expected_value)
    elif sv_single.ndim == 3:
        sv_cls   = sv_single[0, :, target_idx]
        base_val = (explainer.expected_value[target_idx]
                    if hasattr(explainer.expected_value, '__len__')
                    else explainer.expected_value)
    else:
        sv_cls   = sv_single[0]
        base_val = explainer.expected_value

    feat_shap = pd.Series(sv_cls, index=ALL_FEATURES)
    top8      = feat_shap.abs().sort_values(ascending=False).head(8)
    top8_vals = feat_shap[top8.index]

    ax = axes[col_idx]
    bar_colors = ['#C62828' if v > 0 else '#1565C0' for v in top8_vals.values]
    bars = ax.barh(range(8), top8_vals.values, color=bar_colors, alpha=0.85)
    labels = [f'{f}\n= {sample_row.iloc[0][f]:.2f}  [{"derived" if f in DERIVED_FEATURES else "raw"}]'
              for f in top8_vals.index]
    ax.set_yticks(range(8)); ax.set_yticklabels(labels, fontsize=7.5)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP value', fontsize=9)
    ax.set_title(f'{target_name} example\nbase rate = {base_val:.3f}', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, top8_vals.values):
        ax.text(val+(0.001 if val>0 else -0.001), bar.get_y()+bar.get_height()/2,
                f'{val:+.4f}', va='center', ha='left' if val>0 else 'right', fontsize=7)

plt.suptitle('SHAP Waterfall — Top 8 features per severity class\nRed=pushed toward class | Blue=pushed away',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/plots/shap_waterfall_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ SHAP waterfall saved')


✓ SHAP waterfall saved


In [24]:
# ═══════════════════════════════════════════════════
# CELL 19 — Attack signature profiles (derived features)
# ═══════════════════════════════════════════════════
# We include labels from ALL training datasets so the profiles
# show how derived features separate attacks ACROSS datasets.

attack_labels_cic  = ['DoS Hulk', 'DoS GoldenEye', 'PortScan', 'DDoS', 'BENIGN']
attack_labels_unsw = ['DoS', 'Exploits', 'Reconnaissance', 'Normal']
all_attack_labels  = attack_labels_cic + [l for l in attack_labels_unsw
                                           if l not in attack_labels_cic]

derived_to_show = [
    'bwd_fwd_ratio', 'syn_ack_ratio', 'iat_cv',
    'bytes_per_packet', 'pkt_size_asymmetry', 'idle_active_ratio',
]

df_plot   = TRAIN_DF[TRAIN_DF['Label'].isin(all_attack_labels)].copy()
palette   = sns.color_palette('husl', len(all_attack_labels))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes      = axes.flatten()

for ax, feat in zip(axes, derived_to_show):
    data, lbls_present = [], []
    for lbl in all_attack_labels:
        vals = df_plot[df_plot['Label'] == lbl][feat].replace([np.inf,-np.inf], np.nan).dropna()
        if len(vals) < 5:
            continue
        vals = vals.clip(lower=vals.quantile(0.02), upper=vals.quantile(0.98))
        data.append(vals); lbls_present.append(lbl)
    if not data:
        continue
    bp = ax.boxplot(data, labels=lbls_present, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], palette[:len(data)]):
        patch.set_facecolor(color); patch.set_alpha(0.75)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xticklabels(lbls_present, rotation=35, ha='right', fontsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Derived Feature Distributions — CIC-IDS2017 + UNSW-NB15 combined\n'
             'These features expose attack BEHAVIOR, not just raw packet counts',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/plots/derived_feature_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Attack signature profiles saved')


✓ Attack signature profiles saved


In [25]:
# ═══════════════════════════════════════════════════
# CELL 20 — Save explainer + final summary
# ═══════════════════════════════════════════════════
joblib.dump(explainer,        '/kaggle/working/models/shap_explainer.pkl')
joblib.dump(shap_sample,      '/kaggle/working/models/shap_background.pkl')
joblib.dump(DERIVED_FEATURES, '/kaggle/working/models/derived_feature_names.pkl')

print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Best model          : {best_name}')
print(f'RF  macro F1 (CV)   : {rf_cv["test_f1_macro"].mean():.4f} ± {rf_cv["test_f1_macro"].std():.4f}')
print(f'XGB macro F1 (CV)   : {xgb_cv["test_f1_macro"].mean():.4f} ± {xgb_cv["test_f1_macro"].std():.4f}')
print(f'Total features      : {len(ALL_FEATURES)} ({len(available_raw)} raw + {len(DERIVED_FEATURES)} derived)')
print(f'Train datasets      : CIC-IDS2017 + UNSW-NB15')
print(f'Cross-dataset test  : TII-SSRC-23')

print('\nOutput files:')
for root, _, files in os.walk('/kaggle/working'):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath)
        print(f'  {fpath.replace("/kaggle/working/",""):<50} {size/1024:>8.0f} KB')


TRAINING COMPLETE
Best model          : XGBoost
RF  macro F1 (CV)   : 0.9974 ± 0.0002
XGB macro F1 (CV)   : 0.9988 ± 0.0001
Total features      : 50 (38 raw + 12 derived)
Train datasets      : CIC-IDS2017 + UNSW-NB15
Cross-dataset test  : TII-SSRC-23

Output files:
  .virtual_documents/__notebook_source__.ipynb             53 KB
  data/combined_train.csv                              213643 KB
  data/tii_test.csv                                       873 KB
  plots/calibration_curves.png                            176 KB
  plots/cross_dataset_confusion.png                        82 KB
  plots/cross_dataset_metrics.png                          42 KB
  plots/derived_feature_profiles.png                      144 KB
  plots/evaluation_overview.png                           118 KB
  plots/feature_importance_comparison.png                 191 KB
  plots/roc_curves.png                                     68 KB
  plots/shap_beeswarm_high.png                            214 KB
  plots/shap_waterf

In [31]:
import os
import zipfile

# Create a zip of all outputs
output_files = [
    'plots/',
    'models/',
    'data/tii_test.csv',
    '.virtual_documents/__notebook_source__.ipynb'
]

# Zip them
with zipfile.ZipFile('soc_triage_results.zip', 'w') as zipf:
    for file in output_files:
        if os.path.exists(file):
            if os.path.isdir(file):
                for root, dirs, files in os.walk(file):
                    for f in files:
                        zipf.write(os.path.join(root, f))
            else:
                zipf.write(file)

# Download to your local machine
from IPython.display import FileLink
FileLink('soc_triage_results.zip')

/kaggle/working/soc_triage_results.zip